# Deliverable 1: Data Loading and Exploratory Analysis

## Section 1: Setup and Databset Init
Import SparkSession from pyspark.sql, and initialize a SparkSession. Also import all the necessary functions from pyspark.sql.functions and all necessary classes from pyspark.sql.types

In [1]:
import os
import sys

from pyspark.sql.functions import broadcast, explode, split, col, avg, round, desc
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = SparkSession.builder \
    .appName("Week 6 Project 1 Works") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/08 18:18:03 WARN Utils: Your hostname, MacBook-Pro-3.local, resolves to a loopback address: 127.0.0.1; using 192.168.5.102 instead (on interface en0)
26/03/08 18:18:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 18:18:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/08 18:18:04 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
# Create a SparkSession called 'spark'
spark = SparkSession.builder \
    .appName("Week 6 Project 1 Works")\
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")  # Suppress warnings for cleaner output

### Users
This is the given schema for the users dataset in the README:

UserID::Gender::Age::Occupation::Zip-code


In [3]:
schema_users = StructType([
    StructField("UserID", IntegerType(), True),
    StructField("Gender", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("Occupation",IntegerType(), True),
    StructField("Zip-code", StringType(), True)
])

df_users = spark.read \
    .option("header", "false") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("sep", "::") \
    .schema(schema_users) \
    .csv("./ml-1m/users.dat")

Print the schema of the users table:

In [4]:
df_users.printSchema()

root
 |-- UserID: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Occupation: integer (nullable = true)
 |-- Zip-code: string (nullable = true)



How many entries in the users table?

In [5]:
df_users.count()

6040

Show a few rows from the users table:

In [6]:
df_users.show(5)

+------+------+---+----------+--------+
|UserID|Gender|Age|Occupation|Zip-code|
+------+------+---+----------+--------+
|     1|     F|  1|        10|   48067|
|     2|     M| 56|        16|   70072|
|     3|     M| 25|        15|   55117|
|     4|     M| 45|         7|   02460|
|     5|     M| 25|        20|   55455|
+------+------+---+----------+--------+
only showing top 5 rows


### Ratings 
This is the given schema for the ratings dataset in the README:

UserID::MovieID::Rating::Timestamp

In [7]:
schema_ratings  = StructType([
    StructField("UserID", IntegerType(), True),
    StructField("MovieID", IntegerType(), True),
    StructField("Rating", IntegerType(), True),
    StructField("Timestamp",IntegerType(), True)
])

df_ratings = spark.read \
    .option("header", "false") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("sep", "::") \
    .schema(schema_ratings) \
    .csv("./ml-1m/ratings.dat")

df_ratings.show(5)

+------+-------+------+---------+
|UserID|MovieID|Rating|Timestamp|
+------+-------+------+---------+
|     1|   1193|     5|978300760|
|     1|    661|     3|978302109|
|     1|    914|     3|978301968|
|     1|   3408|     4|978300275|
|     1|   2355|     5|978824291|
+------+-------+------+---------+
only showing top 5 rows


Print the schema of the ratings table:

In [8]:
df_ratings.printSchema()

root
 |-- UserID: integer (nullable = true)
 |-- MovieID: integer (nullable = true)
 |-- Rating: integer (nullable = true)
 |-- Timestamp: integer (nullable = true)



How many entries in the ratings table?

In [9]:
df_ratings.count()

1000209

Show a few rows from the ratings table:

In [10]:
df_ratings.show(5)

+------+-------+------+---------+
|UserID|MovieID|Rating|Timestamp|
+------+-------+------+---------+
|     1|   1193|     5|978300760|
|     1|    661|     3|978302109|
|     1|    914|     3|978301968|
|     1|   3408|     4|978300275|
|     1|   2355|     5|978824291|
+------+-------+------+---------+
only showing top 5 rows


### Movies 
This is the given schema for the movies dataset in the README:

MovieID::Title::Genres

In [11]:
schema_movies  = StructType([
    StructField("MovieID", IntegerType(), True),
    StructField("Title", StringType(), True),
    StructField("Genres",StringType(), True)
])

df_movies = spark.read \
    .option("header", "false") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("sep", "::") \
    .schema(schema_movies) \
    .csv("./ml-1m/movies.dat")



Print the schema of the movie table:

In [12]:
df_movies.printSchema()

root
 |-- MovieID: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Genres: string (nullable = true)



How many entries in the moveis table?

In [13]:
df_movies.count()

3883

Show a few rows from the movies table:

In [14]:
df_movies.show(5)

+-------+--------------------+--------------------+
|MovieID|               Title|              Genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Animation|Childre...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|        Comedy|Drama|
|      5|Father of the Bri...|              Comedy|
+-------+--------------------+--------------------+
only showing top 5 rows


# Section 2: Join the Tables
Ratings is joined with Movies on MovieID, and joined with Users on UserID, and the result is stored in df_joined.

In [15]:
df_joined = df_ratings \
    .join(broadcast(df_users), on="UserID", how="inner") \
    .join(broadcast(df_movies), on="MovieID", how="inner")

Print the schema of the joined tables:

In [16]:
df_joined.printSchema()

root
 |-- MovieID: integer (nullable = true)
 |-- UserID: integer (nullable = true)
 |-- Rating: integer (nullable = true)
 |-- Timestamp: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Occupation: integer (nullable = true)
 |-- Zip-code: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Genres: string (nullable = true)



How many columns in the joined table?

In [17]:
len(df_joined.columns)

10

How many entries in the joined table?

In [18]:
df_joined.count()

1000209

Show a few entries from the joined table:

In [19]:
df_joined.show(5)

+-------+------+------+---------+------+---+----------+--------+--------------------+--------------------+
|MovieID|UserID|Rating|Timestamp|Gender|Age|Occupation|Zip-code|               Title|              Genres|
+-------+------+------+---------+------+---+----------+--------+--------------------+--------------------+
|   1193|     1|     5|978300760|     F|  1|        10|   48067|One Flew Over the...|               Drama|
|    661|     1|     3|978302109|     F|  1|        10|   48067|James and the Gia...|Animation|Childre...|
|    914|     1|     3|978301968|     F|  1|        10|   48067| My Fair Lady (1964)|     Musical|Romance|
|   3408|     1|     4|978300275|     F|  1|        10|   48067|Erin Brockovich (...|               Drama|
|   2355|     1|     5|978824291|     F|  1|        10|   48067|Bug's Life, A (1998)|Animation|Childre...|
+-------+------+------+---------+------+---+----------+--------+--------------------+--------------------+
only showing top 5 rows


Explain the join operation:

In [20]:
df_joined.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [MovieID#37, UserID#36, Rating#38, Timestamp#39, Gender#1, Age#2, Occupation#3, Zip-code#4, Title#84, Genres#85]
   +- BroadcastHashJoin [MovieID#37], [MovieID#83], Inner, BuildRight, false
      :- Project [UserID#36, MovieID#37, Rating#38, Timestamp#39, Gender#1, Age#2, Occupation#3, Zip-code#4]
      :  +- BroadcastHashJoin [UserID#36], [UserID#0], Inner, BuildRight, false
      :     :- Filter (isnotnull(UserID#36) AND isnotnull(MovieID#37))
      :     :  +- FileScan csv [UserID#36,MovieID#37,Rating#38,Timestamp#39] Batched: false, DataFilters: [isnotnull(UserID#36), isnotnull(MovieID#37)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/ruts/Projects/cs570_movie-lens_group2/ml-1m/ratings.dat], PartitionFilters: [], PushedFilters: [IsNotNull(UserID), IsNotNull(MovieID)], ReadSchema: struct<UserID:int,MovieID:int,Rating:int,Timestamp:int>
      :     +- BroadcastExchange HashedRelationBroadcastMode(Lis

## Section 3: Basic Statistics

In [ ]:
df_joined.describe().show()

[Stage 23:>                                                         (0 + 1) / 1]

### Observations
- Despite the zipcode being cast as a string rather than an integer, they describe() still calculates a numerical mean and standard deviation. Zipcodes are fixed, location based entries, and have no correlation to each other.
- The Age column is cast as an integer, but it is a categorical variable, as it represents age groups, not distinct ages. At first glance, seeing the minimum age as 1 is odd, but it is a valid entry.
- The Occupation column is cast as an integer, but it is a categorical variable, as it represents occupation groups, not distinct occupations. Executing operations of mean and standard deviation on this column is not valid.

## Section 4: EDA Questions
1. *How many unique genres appear in the dataset? Remember that the Genres column
contains pipe-separated values (e.g., “Action|Comedy”). Count the distinct individual
genres, not the distinct genre combinations.*
2. *What is the average rating given by users in the 25–34 age group? Report to at
least two decimal places.*
3. *Which movie has the most ratings, and how many ratings does it have? Report
both the movie title and the count.*

In [ ]:
# --- Question 1: Unique Genres ---
# Split the genres string by pipe and explode into separate rows
df_genres = df_movies.withColumn("Genre", explode(split(col("Genres"), "\\|")))
unique_genres_count = df_genres.select("Genre").distinct().count()
print(f"A. Number of unique genres: {unique_genres_count}")

# --- Question 2: Average rating for 25-34 age group ---
# In MovieLens 1M dataset, Age=25 represents the 25-34 age bracket
avg_rating_row = (
    df_joined.filter(col("Age") == 25)
    .agg(round(avg("Rating"), 2).alias("AvgRating"))
    .collect()[0]
)
print(f"B. Average rating for 25-34 age group: {avg_rating_row['AvgRating']:.2f}")

# --- Question 3: Movie with the most ratings ---
most_rated_movie = df_joined.groupBy("Title").count().orderBy(desc("count")).first()
print(
    f"C. Movie with most ratings: '{most_rated_movie['Title']}' with {most_rated_movie['count']} ratings"
)

### Question 1. How many unique genres appear in the dataset?
*Answer*: 18 unique genres. 
*Code implementation*: We exploded the pipe-separated Genres column into separate rows using the explode and split functions, then counted the distinct genres.

### Question 2. What is the average rating given by users in the 25–34 age group?
*Answer*: 3.55. 
*Code implementation*: We filtered df_joined for Age == 25 (which represents the 25-34 age bracket in the MovieLens 1M dataset metadata). We aggregated the Rating column using avg() and formatted it to two decimal places using the round() function.

### Question 3. Which movie has the most ratings, and how many ratings does it have?
*Answer*: 'American Beauty (1999)' with 3428 ratings. 
*Code implementation*: We grouped the joined dataset by Title, counted the number of occurrences for each title (representing ratings), ordered by the count in descending order, and took the top result.

## Section 5: Data Quality Observations

### 1. Missing or Null Values
Real-world datasets often have missing entries because a user didn't provide demographic data (like `Zip-code`) or a movie wasn't categorized into a `Genre`. In PySpark, when schemas are explicitly defined or data is loosely cast, invalid formats might also be silently coerced to `null`.

In [ ]:
from pyspark.sql.functions import when, count

# Check for Null values in all columns of df_users
# (Removed isnan() since columns are string or integer)
df_users.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_users.columns
]).show()

# Additionally, check for empty strings in the Zip-code column specifically
empty_zips = df_users.filter(col("Zip-code") == "").count()
print(f"Empty Zip-code strings: {empty_zips}")



### How to handle it
- **For Numerical/Demographic Columns (Age/Occupation)**: If the percentage of missing data is small, you can drop the rows using `df_users.dropna()`. If large, impute values using the median or mean value for numerical sets.

- **For String Columns (Genres/Zip-code)**: Best practice is to replace non-existent values with a placeholder category like `"Unknown"` using `df.fillna("Unknown", subset=["Zip-code"])`.

### 2. Out-of-Bound / Invalid Ratings
Description: the Rating column should be strictly bound between a predefined scale (1 to 5 stars for the MovieLens dataset). Any rating below 1 or above 5 indicates corrupted data or a data entry anomaly that could skew averages and recommendation algorithms.

In [ ]:
# Check for ratings strictly outside the valid 1 to 5 integer range
invalid_ratings = df_ratings.filter((col("Rating") < 1) | (col("Rating") > 5))

invalid_count = invalid_ratings.count()
print(f"Number of invalid outlier ratings: {invalid_count}")

# Option to inspect the corrupted data:
if invalid_count > 0:
    invalid_ratings.show(5)

### How to handle it
- If the invalid ratings are just a small handful of stray values from faulty telemetry/scraping, the safest method is dropping them entirely:
`df_ratings = df_ratings.filter((col("Rating") >= 1) & (col("Rating") <= 5))`

- Alternatively, if we notice a systematic issue (e.g., someone inputted a 0 instead of a 1, or ratings are coming in on a 1-10 scale), we would use withColumn() to apply a mathematical transformation or when().otherwise() clamping logic to fix them programmatically.

# Deliverable 2: Data Cleaning & Feature Exploration

## Section 1: Data Cleaning
Run quick checks for duplicates, referential integrity, out-of-range values, invalid age/occupation codes, and null counts.

### 1a. Quality Checks

#### 1. **Duplicate ratings**: Are there any rows where the same user_id rated the same movie_id more than once?

In [ ]:
from pyspark.sql.functions import col

duplicates = df_ratings.groupBy("UserID", "MovieID").count().filter(col("count") > 1)

duplicates.show(10)
print("Number of duplicate (UserID, MovieID) pairs:", duplicates.count())

**Answer**: No duplicate ratings found. All 1,000,209 rows have unique (UserID, MovieID) combinations.

#### 2. **Referential integrity**: Do all user_id values in the ratings table exist in the users table? Do all movie_id values exist in the movies table? (Use anti-joins to check.)


2.1 Referential integrity: Users

In [ ]:
missing_users = df_ratings.join(df_users.select("UserID"), on="UserID", how="left_anti")

missing_users.show()
missing_users.count()

Answer: 0 ratings contain UserID values that do not exist in the users table.

2.2 Referential integrity: Movies

In [ ]:
missing_movies = df_ratings.join(df_movies.select("MovieID"), on="MovieID", how="left_anti")

missing_movies.show()
missing_movies.count()

Answer: 0 ratings contain MovieID values that do not exist in the movies table.

#### 3. Out-of-range values: 
#### 3.1 Are all ratings between 1 and 5?

In [ ]:
from pyspark.sql.functions import col

out_of_range = df_ratings.filter((col("Rating") < 1) | (col("Rating") > 5))

out_of_range.show()
out_of_range.count()

Answer: 0 ratings are outside the valid 1–5 range.

#### 3.2 Are all age values valid MovieLens age codes (1, 18, 25, 35, 45, 50, 56)?

In [ ]:
df_users.select("Age").distinct().orderBy("Age").show()

Answer: The Age codes present are 1, 18, 25, 35, 45, 50, and 56. All values match the valid MovieLens age categories.

#### 3.3 Are all occupation codes between 0 and 20?

In [ ]:
df_users.select("Occupation").distinct().orderBy("Occupation").show()

Answer: The occupation codes present range from 0 to 20. All values fall within the expected MovieLens occupation categories.

#### 4. Null audit: For each column in your joined DataFrame, report the number of nulls. Show the results in a summary table

In [ ]:
from pyspark.sql.functions import col

null_counts_list = []
for c in df_joined.columns:
    null_counts_list.append((c, df_joined.filter(col(c).isNull()).count()))

spark.createDataFrame(null_counts_list, ["column", "null_count"]).show()

### 1b. Cleaning Actions

#### 1b.1.1 Duplicate ratings (BEFORE)
Check if any (UserID, MovieID) pair appears more than once.

In [ ]:
from pyspark.sql.functions import col

dup_before = df_ratings.groupBy("UserID", "MovieID").count().filter(col("count") > 1)

dup_before.show()
dup_before.count()

#### 1b.1.2 Duplicate ratings (FIX)
If duplicates exist, drop duplicates so each (UserID, MovieID) appears once.

In [ ]:
# Only run this if dup_before.count() was > 0
df_ratings = df_ratings.dropDuplicates(["UserID", "MovieID"])
df_ratings.count()

#### 1b.1.3 Duplicate ratings (AFTER)
Verify that duplicate (UserID, MovieID) pairs are now 0.

In [ ]:
from pyspark.sql.functions import col

dup_after = df_ratings.groupBy("UserID", "MovieID").count().filter(col("count") > 1)

dup_after.show()
dup_after.count()

In [ ]:
df_ratings.count()

Answer: No fix was needed because the BEFORE count was 0.

#### 1b.2.1 Referential integrity(Users)(BEFORE)
Check if any ratings contain a UserID that does not exist in the users table.

In [ ]:
missing_users = df_ratings.join(df_users.select("UserID"), on="UserID", how="left_anti")
missing_users.count()

#### 1b.2.2 Referential integrity (Users)(FIX)
Only use this if the BEFORE count was > 0; drop ratings with UserID values not found in the users table.

In [ ]:
df_ratings = df_ratings.join(df_users.select("UserID"), on="UserID", how="inner")

#### 1b.2.3 Referential integrity (Users)(AFTER)
Verify that UserID count is now 0.

In [ ]:
df_ratings.join(df_users.select("UserID"), on="UserID", how="left_anti").count()

Answer: No fix was needed because the BEFORE count was 0.

#### 1b.3.1 Referential integrity(Movies)(BEFORE)
Check if any ratings contain a MovieID that does not exist in the movies table.

In [ ]:
missing_movies = df_ratings.join(df_movies.select("MovieID"), on="MovieID", how="left_anti")
missing_movies.count()

#### 1b.3.2 Referential integrity (Movies)(FIX)
Only use this if the BEFORE count was > 0; drop ratings with MovieID values not found in the movies table.

In [ ]:
df_ratings = df_ratings.join(df_movies.select("MovieID"), on="MovieID", how="inner")

#### 1b.3.3 Referential integrity (Movies)(AFTER)
Verify that the orphan MovieID count is now 0.

In [ ]:
df_ratings.join(df_movies.select("MovieID"), on="MovieID", how="left_anti").count()

No fix was needed because the BEFORE count was 0.

#### 1b.4.1 Rating range (BEFORE)
Check if any ratings fall outside the valid 1–5 range.

In [ ]:
from pyspark.sql.functions import col

out_of_range = df_ratings.filter((col("Rating") < 1) | (col("Rating") > 5))
out_of_range.count()

#### 1b.4.2 Rating range (FIX)
Only use this if the BEFORE count was > 0; remove ratings outside the 1–5 range.

In [ ]:
df_ratings = df_ratings.filter((col("Rating") >= 1) & (col("Rating") <= 5))

#### 1b.4.3 Rating range (AFTER)
Verify that the out-of-range rating count is now 0.

In [ ]:
df_ratings.filter((col("Rating") < 1) | (col("Rating") > 5)).count()

Answer: No fix was needed because the BEFORE count was 0.

#### 1b.5.1 Null audit (BEFORE)
Count the number of null values in each column of df_joined.

In [ ]:
from pyspark.sql.functions import col

null_counts_before = []
for c in df_joined.columns:
    null_counts_before.append((c, df_joined.filter(col(c).isNull()).count()))

spark.createDataFrame(null_counts_before, ["column", "null_count"]).show()

#### 1b.5.2 Null audit (FIX)
Only use this if any null_count values were > 0; fill missing Genres with "Unknown".

In [ ]:
df_joined = df_joined.fillna({"Genres": "Unknown", "Title": "Unknown"})

#### 1b.5.3 Null audit (AFTER)
Verify the number of null values in each column of df_joined after the fix.

In [ ]:
null_counts_after = []
for c in df_joined.columns:
    null_counts_after.append((c, df_joined.filter(col(c).isNull()).count()))

spark.createDataFrame(null_counts_after, ["column", "null_count"]).show()

Answer: All columns in the joined DataFrame have 0 null values, so no null-handling fix was needed.

#### Rebuild joined table after cleaning
Re-join the cleaned ratings table with users and movies to refresh df_joined.

In [ ]:
df_joined = df_ratings.join(df_users, on="UserID", how="inner").join(df_movies, on="MovieID", how="inner")

df_joined.count()
df_joined.show(5)

## Section 2: Feature Engineering

### 2a. Define Target and Predictors

### Feature Documentation Table

| Group | # | Feature Name | Type | Source | How Computed | Why Potentially Predictive |
|---|---:|---|---|---|---|---|
| Target | 0 | high_rating | Binary (0/1) | Derived from Rating | `when(col("Rating") >= 4, 1).otherwise(0)` | This is the target variable indicating whether a rating is high |
| Predictor | 1 | user_avg_rating | Continuous | Derived from ratings | `groupBy("UserID").avg("Rating")` | Captures each user's typical rating behavior |
| Predictor | 2 | movie_avg_rating | Continuous | Derived from ratings | `groupBy("MovieID").avg("Rating")` | Captures the overall rating tendency of a movie |
| Predictor | 3 | movie_popularity | Integer | Derived from ratings | `groupBy("MovieID").count()` | Frequently rated movies may have more stable rating patterns |
| Predictor | 4 | user_rating_count | Integer | Derived from ratings | `groupBy("UserID").count()` | More active users may show more stable or distinctive rating behavior |
| Predictor | 5 | gender_encoded | Binary (0/1) | Derived from users.dat / Gender | `F -> 1, M -> 0` | User demographics may affect rating preferences |
| Predictor | 6 | num_genres | Integer | Derived from movies.dat / Genres | `size(split(col("Genres"), "\\"))` | Multi-genre movies may appeal differently than single-genre movies |
| Predictor | 7 | release_year | Integer | Derived from movies.dat / Title | `regexp_extract(col("Title"), r"\\((\\d{4})\\)", 1).cast("int")` | Movie era may influence audience preferences and ratings |
| Predictor | 8 | movie_age | Integer | Derived from release_year | `2000 - release_year` | Older vs newer movies may receive different rating patterns |

### Optional reset block for reruns:
If these engineered columns already exist from an earlier notebook run, drop them first so the feature-creation cells can be re-executed cleanly.


In [ ]:
from pyspark.sql.functions import col, when, split, size, regexp_extract, length
cols_to_drop = [
    "high_rating", "user_avg_rating", "movie_avg_rating", "movie_popularity", "user_rating_count",
    "gender_encoded", "num_genres", "release_year", "movie_age"
]

for column in cols_to_drop:
    if column in df_joined.columns:
        df_joined = df_joined.drop(column)

df_joined.printSchema()

#### 0) Target: high_rating

In [ ]:
df_joined = df_joined.withColumn("high_rating", when(col("Rating") >= 4, 1).otherwise(0))

#### 1) user_avg_rating

In [ ]:
df_joined = df_joined.join(df_joined.groupBy("UserID").agg({"Rating":"avg"}).withColumnRenamed("avg(Rating)", "user_avg_rating"), on="UserID", how="left")

#### 2) movie_avg_rating

In [ ]:
movie_avg = df_joined.groupBy("MovieID").agg(avg("Rating").alias("movie_avg_rating"))
df_joined = df_joined.join(movie_avg, on="MovieID", how="left")

#### 3) movie_popularity (ratings count per movie)

In [ ]:
movie_pop = df_joined.groupBy("MovieID").agg(count("*").alias("movie_popularity"))
df_joined = df_joined.join(movie_pop, on="MovieID", how="left")

#### 4) user_rating_count (ratings count per user)


In [ ]:
user_cnt = df_joined.groupBy("UserID").agg(count("*").alias("user_rating_count"))
df_joined = df_joined.join(user_cnt, on="UserID", how="left")

#### 5) gender_encoded (F=1, M=0)


In [ ]:
df_joined = df_joined.withColumn("gender_encoded", when(col("Gender") == "F", 1).otherwise(0))

#### 6) num_genres


In [ ]:
df_joined = df_joined.withColumn("num_genres", size(split(col("Genres"), "\\|")))

#### 7) release_year (extract from Title)


In [ ]:
year_str = regexp_extract(col("Title"), r"\((\d{4})\)", 1)
df_joined = df_joined.withColumn("release_year", when(length(year_str) == 4, year_str.cast("int")).otherwise(None))

#### 8) movie_age (relative to year 2000)


In [ ]:
df_joined = df_joined.withColumn("movie_age", when(col("release_year").isNotNull(), 2000 - col("release_year")).otherwise(None))

### Verification

In [ ]:
df_joined.select(
    "UserID","MovieID","Rating","high_rating",
    "user_avg_rating","movie_avg_rating","movie_popularity","user_rating_count",
    "gender_encoded","num_genres","release_year","movie_age"
).show(5)

df_joined.select(
    "high_rating","user_avg_rating","movie_avg_rating","movie_popularity","user_rating_count",
    "gender_encoded","num_genres","release_year","movie_age"
).describe().show()

 ### 2b Feature 1: release_year

In [ ]:
from pyspark.sql import functions as F

df_features = df_joined.withColumn(
    "release_year",
    F.regexp_extract(F.col("Title"), r"\((\d{4})\)", 1).cast("int")
)

df_features.select("Title", "release_year").show(10, truncate=False)
df_features.select("release_year").describe().show()

  release_year extracts the movie’s release year from the Title field. This can be predictive because rating behavior often differs across eras (older classics vs newer releases).

 ### 2b Feature 2: movie_age

In [ ]:
df_features = df_features.withColumn(
    "movie_age",
    F.lit(2000) - F.col("release_year")
)

df_features.select("Title", "release_year", "movie_age").show(10, truncate=False)
df_features.select("movie_age").describe().show()

 movie_age represents how old a movie was around the time this dataset was collected (~2000). It may capture trends where newer movies receive different ratings than older movies.

### 2b Feature 3 : primary_genre

In [ ]:
df_features = df_features.withColumn(
    "primary_genre",
    F.split(F.col("Genres"), r"\|").getItem(0)
)

df_features.select("Genres", "primary_genre").show(10, truncate=False)
df_features.groupBy("primary_genre").count().orderBy(F.desc("count")).show(10, truncate=False)

primary_genre takes the first genre listed as a simplified categorical label. This reduces genre complexity and helps the model learn broad preference patterns across users.

## Section 3: Exploratory Data Analysis

### 3a. Target Variable Distribution

Class balance of ``high_ratings`` (counts & percentages)

In [ ]:
### Evaluate class balance for high_rating
print("\n--- Class Balance for 'high_rating' ---")
class_counts = df_joined.groupBy("high_rating").count().collect()
total_ratings = sum(row["count"] for row in class_counts)

labels = []
counts = []
for row in class_counts:
    rating_class = row["high_rating"]
    c = row["count"]
    pct = (c / total_ratings) * 100
    print(f"Class '{rating_class}': {c} ratings ({pct:.2f}%)")
    labels.append(str(rating_class))
    counts.append(c)

Bar chart visualization of class balance of ``high_ratings``

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
bars = plt.bar(labels, counts, color=["skyblue", "lightgreen"])
plt.title("Class Balance of 'high_rating'")
plt.xlabel("high_rating (0 = <4, 1 = >=4)")
plt.ylabel("Number of Ratings")
plt.xticks(labels, [f"{l} (High)" if l == "1" else f"{l} (Low)" for l in labels])

# Add percentage labels on top of bars
for bar in bars:
    yval = bar.get_height()
    pct = (yval / total_ratings) * 100
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        yval + (total_ratings * 0.01),
        f"{pct:.2f}%",
        ha="center",
        va="bottom",
    )

plt.tight_layout()

**Answer**: The dataset is somewhat imbalanced but not severely. 'High' ratings (1) make up 57.52% of the dataset, while 'Low' ratings (0) make up 42.48%.

### 3b. Numerical Feature Distributions

In [ ]:
# Extract the unique values for plotting
user_avg_df = df_joined.select("UserID", "user_avg_rating").dropDuplicates()
user_avg_rows = user_avg_df.collect()
user_avg_data = [
    r["user_avg_rating"] for r in user_avg_rows if r["user_avg_rating"] is not None
]

movie_avg_df = df_joined.select(
    "MovieID", "movie_avg_rating", "movie_popularity"
).dropDuplicates()
movie_rows = movie_avg_df.collect()
movie_avg_data = [
    r["movie_avg_rating"] for r in movie_rows if r["movie_avg_rating"] is not None
]
movie_pop_data = [
    r["movie_popularity"] for r in movie_rows if r["movie_popularity"] is not None
]

# Create subplots
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# Histogram for User Average Rating
axs[0].hist(user_avg_data, bins=30, color="coral", edgecolor="black")
axs[0].set_title("Distribution of User Avg Rating")
axs[0].set_xlabel("Average Rating")
axs[0].set_ylabel("Frequency")

# Histogram for Movie Average Rating
axs[1].hist(movie_avg_data, bins=30, color="lightgreen", edgecolor="black")
axs[1].set_title("Distribution of Movie Avg Rating")
axs[1].set_xlabel("Average Rating")
axs[1].set_ylabel("Frequency")

# Histogram for Movie Popularity
axs[2].hist(movie_pop_data, bins=50, color="skyblue", edgecolor="black")
axs[2].set_title("Distribution of Movie Popularity (Rating Count)")
axs[2].set_xlabel("Number of Ratings")
axs[2].set_ylabel("Frequency")

plt.tight_layout()
plt.savefig("numerical_features_histograms.png")

1. **Are any features heavily skewed?** Yes, ``movie_popularity`` (the number of ratings a movie receives) is extremely right-skewed. The vast majority of movies have a very small number of ratings, while a few blockbuster movies have thousands of ratings. The ``user_avg_rating`` and ``movie_avg_rating`` features appear roughly normally distributed but have a slight left-skew, as most average ratings tend to center between 3.2 and 3.7.

2. **Are there outliers?** (e.g., movies with very few or very many ratings) Yes, there are major outliers, particularly in ``movie_popularity``.
    - There are massive outlier movies with over 3,000 ratings (the maximum count is 3,428 ratings for a single movie).
    - Meanwhile, the mean (average) number of ratings per movie is only about ~270, and there is a long tail of movies with the absolute minimum of 1 rating.

### 3c. Categorical vs. Target: Group Comparisons
Comparing the average high_rating rate (or average rating) across:
- Gender vs. High Rating Rate
- Age Group vs. High Rating Rate
- Top 10 Genres vs. Avg Rating
- Top 10 Occupations by Avg Rating

In [ ]:
# 1. Gender vs. high_rating rate
gender_stats = (
    df_joined.groupBy("Gender")
    .agg(avg("high_rating").alias("avg_high_rating"))
    .collect()
)
gender_labels = [row["Gender"] for row in gender_stats]
gender_rates = [row["avg_high_rating"] for row in gender_stats]

print("Gender vs. high_rating:")
for r in gender_stats:
    print(f"  Gender {r['Gender']}: {r['avg_high_rating']:.4f} rate of high ratings")

# 2. Age group vs. high_rating rate
age_stats = (
    df_joined.groupBy("Age")
    .agg(avg("high_rating").alias("avg_high_rating"))
    .orderBy("Age")
    .collect()
)
age_labels = [str(row["Age"]) for row in age_stats]
age_rates = [row["avg_high_rating"] for row in age_stats]

print("\nAge vs. high_rating:")
for r in age_stats:
    print(f"  Age {r['Age']}: {r['avg_high_rating']:.4f} rate of high ratings")

# 3. Occupation vs. average rating (Top 10)
occ_stats = (
    df_joined.groupBy("Occupation")
    .agg(avg("Rating").alias("avg_rating"))
    .orderBy(col("avg_rating").desc())
    .collect()
)
occ_labels = [str(row["Occupation"]) for row in occ_stats[:10]]
occ_rates = [row["avg_rating"] for row in occ_stats[:10]]

print("\nTop Occupations by average rating:")
for r in occ_stats[:5]:
    print(f"  Occupation {r['Occupation']}: {r['avg_rating']:.4f} average rating")

# 4. Top 10 genres vs. average rating
df_genres = df_joined.withColumn("Genre", explode(split(col("Genres"), r"\|")))
genre_stats = df_genres.groupBy("Genre").agg(
    avg("Rating").alias("avg_rating"), count("*").alias("count")
)
# Filter out genres with very few ratings to get a meaningful top 10
genre_stats_filtered = (
    genre_stats.filter(col("count") > 100).orderBy(col("avg_rating").desc()).collect()
)

genre_labels = [row["Genre"] for row in genre_stats_filtered[:10]]
genre_rates = [row["avg_rating"] for row in genre_stats_filtered[:10]]

print("\nTop 10 Genres by average rating:")
for r in genre_stats_filtered[:10]:
    print(f"  Genre {r['Genre']}: {r['avg_rating']:.4f} average rating")

Visualizations of the above comparisons

In [ ]:
# Visualizing Categorical Comparisons
fig, axs = plt.subplots(2, 2, figsize=(18, 12))
axs = axs.flatten()

# Gender Bar Chart
axs[0].bar(
    gender_labels,
    gender_rates,
    color=["pink", "lightblue"] if len(gender_labels) == 2 else "skyblue",
)
axs[0].set_title("Gender vs. High Rating Rate")
axs[0].set_xlabel("Gender")
axs[0].set_ylabel('Avg Rate of "High" Ratings')
axs[0].set_ylim(0.5, 0.65)  # Zoom in to see the difference

# Age Bar Chart
axs[1].bar(age_labels, age_rates, color="orange", edgecolor="black")
axs[1].set_title("Age Group vs. High Rating Rate")
axs[1].set_xlabel("Age Group")
axs[1].set_ylabel('Avg Rate of "High" Ratings')
axs[1].set_ylim(0.5, 0.70)

# Occupation Bar Chart (Top 10)
axs[2].bar(occ_labels, occ_rates, color="purple", edgecolor="black")
axs[2].set_title("Top 10 Occupations by Avg Rating")
axs[2].set_xlabel("Occupation ID")
axs[2].set_ylabel("Average Rating")
axs[2].set_ylim(3.5, 4.0)

# Genre Bar Chart (Top 10)
axs[3].bar(genre_labels, genre_rates, color="teal", edgecolor="black")
axs[3].set_title("Top 10 Genres by Avg Rating")
axs[3].set_xlabel("Genre")
axs[3].set_ylabel("Average Rating")
axs[3].set_ylim(3.5, 4.2)
for tick in axs[3].get_xticklabels():
    tick.set_rotation(45)

plt.tight_layout()
plt.savefig("categorical_features_analysis.png")

1. *Gender vs. high_rating rate: Do men or women give "high" ratings more often?* **Answer:** Women (Gender F) give "high" ratings slightly more often (approx. 59.07% rate) compared to men (approx. 57.01%).

2. *Age group vs. high_rating rate: Which age group is most generous?* **Answer:** Older users are the most generous. The 56+ age group has the highest rate of "high" ratings (64.63%), followed closely by the 50-55 age bracket (62.19%). Generally, as the age bracket goes up, the percentage of high ratings increases.

3. *Occupation vs. average rating: Do certain occupations rate differently?* **Answer:** Yes, opinions do vary slightly by occupation, though the differences are quite narrow. The highest average ratings tend to hover around 3.7 to 3.8 stars. For example, occupations like 13 (retired) and 15 (scientist) might rate movies marginally higher overall than the baseline average.

4. *Top 10 genres vs. average rating: Which genres get the highest ratings?* **Answer:** Film-Noir receives the highest average ratings by far (approx. 4.07 average stars), followed by Documentary (3.93) and War (3.89) movies. Other top genres included Drama (3.77) and Crime (3.71).

### 3d. Numerical Feature Correlations

In [ ]:
import seaborn as sns

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

# Assemble numerical features into a vector
num_features = ["user_avg_rating", "movie_avg_rating",
"movie_popularity", "age", "num_genres"]
assembler = VectorAssembler(inputCols=num_features, outputCol="num_vec")
vec_df = assembler.transform(df_joined).select("num_vec")

# Compute Pearson correlation matrix
corr_matrix = Correlation.corr(vec_df, "num_vec").head()[0]
corr_array = corr_matrix.toArray()

# Visualize
plt.figure(figsize=(8, 6))
sns.heatmap(corr_array, annot=True, fmt=".2f",
xticklabels=num_features, yticklabels=num_features,
cmap="coolwarm", center=0)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

1. *Which pair of features has the strongest correlation?* **Answer:** The strongest correlation among the independent features is between ``movie_avg_rating`` and ``movie_popularity``, with a correlation coefficient of 0.5219. This makes intuitive sense: movies that are more popular (have more ratings) tend to have higher average ratings (since highly-rated movies attract more viewers).

2. *Are any features redundant (correlation > 0.8)? If so, would you drop one?* **Answer:** No features are redundant. The highest correlation between any two independent variables is only 0.52 (as mentioned above). None of the features cross the typical threshold for multicollinearity/redundancy > 0.8. Therefore, I would not drop any of them based on redundancy.

3. *Which features correlate most strongly with rating?* **Answer:** When comparing against the target ``Rating`` itself, the strongest correlations are:
    - ``movie_avg_rating`` (0.4885) — The movie's historic average rating is the strongest individual predictor of what any single rating will be.
    - ``user_avg_rating`` (0.3904) — How generously the user tends to rate is the second strongest predictor.
    - ``movie_popularity`` (0.2550) — Popularity also shows a moderate positive correlation with individual ratings.

Features like ``Age`` (0.0569) and ``num_genres`` (0.0020) have very weak correlations with the direct ``Rating``.

### 3e. Categorical vs. Categorical 

- Genre preferences by gender: Do men and women prefer different genres?
- Genre preferences by age group: Do younger vs. older users watch different genres?

In [ ]:
### Categorical-Categorical Combinations
print("---Genre Preferences by Gender---")
# Let's count genre preferences by Gender
gender_genre = df_genres.groupBy("Gender", "Genre").agg(count("*").alias("count"))
gender_totals = df_genres.groupBy("Gender").agg(count("*").alias("total"))
gender_genre = gender_genre.join(gender_totals, on="Gender")
gender_genre = gender_genre.withColumn("pct", col("count") / col("total") * 100)

for g in ["F", "M"]:
    print(f"Top 5 Genres for Gender {g}:")
    top = gender_genre.filter(col("Gender") == g).orderBy(col("pct").desc()).collect()
    for r in top[:5]:
        print(f"  {r['Genre']}: {r['pct']:.2f}%")

print("\n---Genre Preferences by Age Group---")
age_genre = df_genres.groupBy("Age", "Genre").agg(count("*").alias("count"))
age_totals = df_genres.groupBy("Age").agg(count("*").alias("total"))
age_genre = age_genre.join(age_totals, on="Age")
age_genre = age_genre.withColumn("pct", col("count") / col("total") * 100)

# Comparing very young (Age 1) vs general adult (Age 25) vs older (Age 56)
for a in [1, 25, 56]:
    age_label = "Under 18" if a == 1 else ("25-34" if a == 25 else "56+")
    print(f"Top 5 Genres for Age Group {age_label} (Code {a}):")
    top = age_genre.filter(col("Age") == a).orderBy(col("pct").desc()).collect()
    for r in top[:5]:
        print(f"  {r['Genre']}: {r['pct']:.2f}%")

1. *Genre preferences by gender: Do men and women prefer different genres?* **Answer:** Yes, slightly. While both genders share heavily in watching Drama and Comedy blockbusters, their downstream preferences diverge:
    - **Men have Action (13.27%) and Sci-Fi (8.14%)** higher up in their top 5 most frequently rated genres.
    - **Women rate Romance (9.94%)** heavily, putting it in their top 5, whereas it does not make the top 5 for men. Conversely, Sci-Fi falls out of the top 5 for women.
2. *Genre preferences by age group: Do younger vs. older users watch different genres?* **Answer:** Yes, there is a very clear generational shift in preferences.
    - **Younger users (Under 18):** Rate Comedy (18.98%) as their most frequent genre, and Children's movies comfortably make their top 5 (7.37%).
    - **Young Adults (25-34):** Have a more balanced mix, with Comedy (17.23%) and Drama (16.68%) nearly tied, and Sci-Fi entering their top 5.
    - **Older users (56+):** Overwhelmingly prefer Drama as their absolute favorite (21.91%). Romance enters their top 5, while Children's and Sci-Fi movies fall completely out of the top echelon for this age group.

## 3f. Feature-Target Summary

| Feature                   | Type        | Relationship to Target                   | Strength | Keep for D4? |
|---------------------------|-------------|------------------------------------------|----------|--------------|
| ``movie_avg_rating``          | Numerical   | Strong Positive Correlation (0.52)       | Strong   | Yes          |
| ``user_avg_rating``           | Numerical   | Strong Positive Correlation (0.39)       | Strong   | Yes          |
| ``movie_popularity``          | Numerical   | Moderate Positive Correlation (0.26)     | Moderate | Yes          |
| ``genre``                    | Categorical | High average ratings (>3.9)              | Moderate | Yes          |
| ``age``                       | Categorical | Older users rate slightly higher (~0.64) | Weak     | Maybe        |
| ``gender``            | Binary      | Small difference (~0.02) in high rating  | Weak     | Maybe        |
| ``occupation``                | Categorical | Very narrow differences in avg ratings   | Weak     | Maybe        |

For D4:
- ``movie_avg_rating``: A movie's historical average is highly predictive of any individual future rating (the wisdom of the crowd effect).
- ``user_avg_rating``: Explains baseline user behavior. A user's historical tendency to be a "harsh" or "generous" critic anchors their future ratings.
- ``movie_popularity`` (rating count): Well-known, highly rated movies attract more positive ratings (popularity bias usually heavily correlates with a high_rating).
- ``genre``: Knowing whether a movie is a Drama or Film-Noir vs. an Action or Children's movie provides a strong baseline expectation for the overall rating quality.

The rest of the user demographic data (Age, Gender, Occupation) showed relatively weak correlation and narrow variance.


## Section 4: Reflection

### Cleaning summary: What quality issues did you find (or not find)? Was anything surprising about the data quality?
The data quality of the MovieLens 1M dataset was surprisingly high. We found no unexpected missing values (nulls) in critical columns, and referential integrity between the tables was perfectly maintained out of the box. The only minor issues were some movies without proper genre tags, but overall, it lacked the severe corruption (like negative ratings, wrong data types, or orphaned records) often found in real-world raw datasets.

### Feature insights: Which feature do you think will be the strongest predictor of high ratings? What evidence from your EDA supports this?
The strongest predictor of high ratings will be the `movie_avg_rating`. In our EDA, this feature showed the highest Pearson correlation with individual ratings (0.49). Intuitively, a generally well-regarded movie is highly likely to continue receiving high ratings, demonstrating a strong "wisdom of the crowd" effect.

### Surprises: Did any feature behave differently than you expected? (e.g., a feature you thought would matter but doesn't, or vice versa)
The low impact of user demographics (Age, Gender, Occupation) was somewhat surprising. We might assume that rating behavior would vary wildly across different age groups or occupations, but the correlation matrix showed these features had near-zero linear correlation (e.g., Age vs Rating is only 0.05). While there are slight categorical differences (e.g., older users being slightly more generous), demographics are heavily overshadowed by the movie's inherent quality.

### Next steps: What additional features or transformations would you want to try in D4? List at least 2 ideas.
1. **Timestamp Features**: We could extract the day of the week, month, or time of day from the `Timestamp` to see if users rate more harshly on certain days (e.g., Mondays vs Weekends).
2. **Genre Encoding & Interactions**: Instead of just a primary genre or a count, we could deeply one-hot encode all genres and create interaction terms to capture personalized genre preferences.

### Limitations: What are the limitations of this dataset for building a movie recommender? (Think about what data is missing that would help.)
A major limitation is the lack of rich content features about the movies themselves, such as the director, lead actors, budget, or plot summary text. Without these, the model relies heavily on collaborative filtering signals. Additionally, the dataset is quite old (up to the year 2000), meaning it lacks modern movies and cannot capture shifting generic trends over the last two decades.
